# Day 3: Protein Mix Spike-Ins

In [1]:
!pip install nucleus-cdk==0.5.0rc2 | tail -n2

In [2]:
from cdk.analysis.cytosol import platereader as pr
import matplotlib.pyplot as plt
import pandas as pd
import warnings

# Ignore warnings
warnings.filterwarnings('ignore')

# Initialize plotting
pr.plot_setup()

# Load the data

Provide a CSV file containing the data, and a platemap. This function returns both the data with the plate map mapped to it, and the platemap by itself, which is useful for certain tasks.

In [ ]:
platemap_path = "../1-design/20250514-OPWS-blueteam.tsv"
data_path = "../2-data/20250514-211604-pure-timecourse-gfp-nucleus-pure-workshop-day3-protein-mix-debug-biotek-cdk.txt"


data, platemap = pr.load_platereader_data(data_path, platemap_path)

# Well D5 ("OP -", replicate 3/3) fits to k~=0 (degenerate/non-responding), which crashes kinetic_analysis's lag-time calculation. Drop it; keep B5/C5 (its two other replicates).
# Also exclude Type=="Standard" (plamGFP dilution/calibration curve) - these are static purified-protein standards, not live PURE reactions, so sigmoid growth kinetics isn't meaningful for them and several degenerate the same way. Controls (Type=="Control") are kept.
data_drop = data[(data["Well"] != "D5") & (data["Type"] != "Standard")]

platemap

# Basic Plots

## Kinetics 
Kinetic time traces of every well on the plate

In [ ]:
#| label: fig:day3-kinetics-controls

ctrl_data = data_drop[data_drop["Type"]=="Control"]

names_to_remove_0 = ['Ribo',
                   #'NEB +',
                   # 'NEB -',
                   # 'Ribo'
                  ]

replace_dict_0 = {'OP +':'PM 6 +',
                  'OP -':'PM 6 -',
                  'NEB +': 'Positive',
                  'NEB -': 'Negative',
                  }

custom_order_0 = ['PM 6 +', 
                  'PM 6 -',
                  'Positive', 
                  'Negative']

color_map_0 = {'PM 6 +':'#2ca02c',
             'PM 6 -':'#1f77b4',
             'Positive':'#90EE90',  
             'Negative':'#87CEEB'
               }

data_drop_ctrl_0 = ctrl_data.drop(ctrl_data[ctrl_data['Name'].isin(names_to_remove_0)].index)
data_drop_ctrl_0['Name'] = data_drop_ctrl_0['Name'].replace(replace_dict_0)


pr.plot_curves(data_drop_ctrl_0, palette=color_map_0, hue_order=custom_order_0);
plt.yscale('log')

In [5]:
#| label: fig:day3-steadystate-controls

pr.plot_steadystate(data_drop_ctrl_0, order=list(color_map_0.keys()), palette=color_map_0);
plt.yscale('log')
plt.axhline(y=443, color='black', linestyle='--', alpha=0.5);

OutOfBoundsTimedelta: Cannot cast -inf from h to 'ns' without overflow.

In [ ]:
#| label: fig:day3-kinetics-argrs

argrs_data = data[(~pd.isna(data["ArgRS"]) & (data["ArgRS"] > 0.5))]
argrs_ctrl_data = data[(~pd.isna(data["ArgRS (ctrl)"]) & (data["ArgRS (ctrl)"] > 0.5))]

argsrs_comb_data = pd.concat([argrs_data, argrs_ctrl_data, data_drop_ctrl_0])

# names_to_remove = ['ArgRS 1 uL',
#                    'ArgRS 1.5 uL'
#                   ]

# replace_dict = {'ArgsRS (ctrl) 1.0 uL':'ArgRS 1.0 uL',
#                 'ArgRS (ctrl) 1.5 uL':'ArgRS 1.5 uL',
#                 'OP +':'PM 6 +',
#                 'OP -':'PM 6 -'}

names_to_remove_1 = ['ArgsRS (ctrl) 1.0 uL',
                   'ArgRS (ctrl) 1.5 uL',
                   'Positive',
                   'Negative'
                  ]

replace_dict_1 = {'ArgRS 1 uL':'ArgRS 1.0 uL',
                'ArgRS 1.5 uL':'ArgRS 1.5 uL',
                'OP +':'PM 6 +',
                'OP -':'PM 6 -'}

custom_order_1 = ['ArgRS 1.0 uL', 
                'ArgRS 1.5 uL',
                'PM 6 +', 
                'PM 6 -']

color_map_1 = {'ArgRS 1.0 uL':'#d62728',
             'ArgRS 1.5 uL':'#ff7f0e',
             'PM 6 +':'#2ca02c',
             'PM 6 -':'#1f77b4'
               }

argsrs_comb_data_drop = argsrs_comb_data.drop(argsrs_comb_data[argsrs_comb_data['Name'].isin(names_to_remove_1)].index)
argsrs_comb_data_drop['Name'] = argsrs_comb_data_drop['Name'].replace(replace_dict_1)

argrs_plot = pr.plot_curves(argsrs_comb_data_drop, palette=color_map_1);
argrs_plot.set(ylim=(0, 1400));

# pr.plot_curves(argsrs_comb_data);

In [ ]:
#| label: fig:day3-kinetics-t7rnap

t7rnap_data = data[(~pd.isna(data["T7 RNAP"]) & (data["T7 RNAP"] > 0.5))]
t7rnap_ctrl_data = data[(~pd.isna(data["T7 RNAP (ctrl)"]) & (data["T7 RNAP (ctrl)"] > 0.5))]

t7rnap_comb_data = pd.concat([t7rnap_data, t7rnap_ctrl_data, data_drop_ctrl_0])

# names_to_remove = ['T7 RNAP 1 uL',
#                    'T7 RNAP 1.5 uL',
#                    '1 uL T7 RNAP + 1 uL EF-TU (ctrl)'
#                   ]


# replace_dict = {'T7 RNAP (ctrl) 1.0 uL':'T7 RNAP 1.0 uL',
#                 'T7 RNAP (ctrl) 1.5 uL':'T7 RNAP 1.5 uL',
#                 'OP +':'PM 6 +',
#                 'OP -':'PM 6 -'}

names_to_remove_2 = ['T7 RNAP (ctrl) 1.0 uL',
                   'T7 RNAP (ctrl) 1.5 uL',
                   '1 uL T7 RNAP + 1 uL EF-TU (ctrl)',
                   'Positive',
                   'Negative'
                  ]


replace_dict_2 = {'T7 RNAP 1 uL':'T7 RNAP 1.0 uL',
                'T7 RNAP 1.5 uL':'T7 RNAP 1.5 uL',
                'OP +':'PM 6 +',
                'OP -':'PM 6 -'}

custom_order_2 = ['T7 RNAP (ctrl) 1.0 uL', 
                'T7 RNAP (ctrl) 1.5 uL',
                'PM 6 +', 
                'PM 6 -']

color_map_2 = {'T7 RNAP 1.0 uL':'#d62728',
             'T7 RNAP 1.5 uL':'#ff7f0e',
             'PM 6 +':'#2ca02c',
             'PM 6 -':'#1f77b4'
               }

t7rnap_comb_data_drop = t7rnap_comb_data.drop(t7rnap_comb_data[t7rnap_comb_data['Name'].isin(names_to_remove_2)].index)
t7rnap_comb_data_drop['Name'] = t7rnap_comb_data_drop['Name'].replace(replace_dict_2)

t7rnap_plot = pr.plot_curves(t7rnap_comb_data_drop, palette=color_map_2);
t7rnap_plot.set(ylim=(0, 1400));

In [ ]:
#| label: fig:day3-kinetics-if2

if2_data = data[(~pd.isna(data["IF2"]) & (data["IF2"] > 0.5))]
if2_ctrl_data = data[(~pd.isna(data["IF2 (ctrl)"]) & (data["IF2 (ctrl)"] > 0.5))]

if2_comb_data = pd.concat([if2_data, if2_ctrl_data, data_drop_ctrl_0])

# names_to_remove = ['IF2 1 uL',
#                    'IF 1.5 uL',
#                    '1 uL IF2 + 1 uL EF-TU'
#                   ]

names_to_remove_3 = ['IF2 (ctrl) 1.0 uL',
                   'IF2 (ctrl) 1.5 uL',
                   '1 uL IF2 + 1 uL EF-TU',
                   'Positive',
                   'Negative'
                  ]

# replace_dict = {'IF2 (ctrl) 1.0 uL':'IF2 1.0 uL',
#                 'IF2 (ctrl) 1.5 uL':'IF2 1.5 uL',
#                 'OP +':'PM 6 +',
#                 'OP -':'PM 6 -'}

replace_dict_3 = {'IF2 1 uL':'IF2 1.0 uL',
                'IF 1.5 uL':'IF2 1.5 uL',
                'OP +':'PM 6 +',
                'OP -':'PM 6 -'}

custom_order_3 = ['IF2 1.0 uL', 
                'IF2 1.5 uL',
                'PM 6 +', 
                'PM 6 -']

color_map_3 = {'IF2 1.0 uL':'#d62728',
             'IF2 1.5 uL':'#ff7f0e',
             # '1 uL IF2 + 1 uL EF-TU': 'Red',
             'PM 6 +':'#2ca02c',
             'PM 6 -':'#1f77b4'
               }

if2_comb_data_drop = if2_comb_data.drop(if2_comb_data[if2_comb_data['Name'].isin(names_to_remove_3)].index)
if2_comb_data_drop['Name'] = if2_comb_data_drop['Name'].replace(replace_dict_3)

if2_plot = pr.plot_curves(if2_comb_data_drop, palette=color_map_3);
if2_plot.set(ylim=(0, 1400));

In [ ]:
#| label: fig:day3-kinetics-eftu

eftu_data = data[(~pd.isna(data["EF-TU"]) & (data["EF-TU"] > 0.5))]
eftu_ctrl_data = data[(~pd.isna(data["EF-TU (ctrl)"]) & (data["EF-TU (ctrl)"] > 0.5))]

eftu_comb_data = pd.concat([eftu_data, eftu_ctrl_data, data_drop_ctrl_0])

names_to_remove_4 = ['EF-TU 1 uL',
                   'EF-TU 1.5 uL',
                   '1 uL IF2 + 1 uL EF-TU',
                   '1 uL T7 RNAP + 1 uL EF-TU (ctrl)',
                   'Positive',
                   'Negative'
                  ]

replace_dict_4 = {'EF-TU (ctrl) 1.0 uL':'EF-TU 1.0 uL',
                'EF-TU (ctrl) 1.5 uL':'EF-TU 1.5 uL',
                'OP +':'PM 6 +',
                'OP -':'PM 6 -'}

custom_order_4 = ['EF-TU 1.0 uL', 
                'EF-TU 1.5 uL',
                'PM 6 +', 
                'PM 6 -']

color_map_4 = {'EF-TU 1.0 uL':'#d62728',
             'EF-TU 1.5 uL':'#ff7f0e',
             'PM 6 +':'#2ca02c',
             'PM 6 -':'#1f77b4'
               }

eftu_comb_data_drop = eftu_comb_data.drop(eftu_comb_data[eftu_comb_data['Name'].isin(names_to_remove_4)].index)
eftu_comb_data_drop['Name'] = eftu_comb_data_drop['Name'].replace(replace_dict_4)

eftu_plot = pr.plot_curves(eftu_comb_data_drop, palette=color_map_4);
eftu_plot.set(ylim=(0, 1400));

In [ ]:
#| label: fig:day3-kinetics-pairwise

pair_data = data[data["Type"]=="Pairwise"]
pair_comb_data = pd.concat([pair_data, data_drop_ctrl_0])

names_to_remove_5 = ['1 uL IF2 + 1 uL EF-TU',
                   '0.5 uL T7 RNAP + 0.5 uL IF2 ',
                   'Positive',
                   'Negative'
                  ]

replace_dict_5 = {'0.5 uL T7 RNAP + 0.5 uL EF-TU (ctrl)':'Mixture 1',
                '0.5 uL IF2 + 0.5 uL EF-TU (ctrl)':'Mixture 2',
                '1 uL T7 RNAP + 1 uL EF-TU (ctrl)': 'Mixture 3',
                '0.5 uL ArgRS + 0.5 uL T7 RNAP + 0.5 uL IF 2 + 0.5 uL EF-TU (ctrl)': 'Mixture 4',
                'OP +':'PM 6 +',
                'OP -':'PM 6 -'}

custom_order_5 = ['0.5 uL T7 RNAP + 0.5 uL EF-TU', 
                '1 uL T7 RNAP + 1 uL EF-TU',
                '0.5 uL IF2 + 0.5 uL EF-TU',
                '0.5 uL ArgRS + 0.5 uL T7 RNAP + 0.5 uL IF 2 + 0.5 uL EF-TU',
                'PM 6 +', 
                'PM 6 -']

color_map_5 = {'Mixture 1': '#e377c2',     # Pink/Magenta
             'Mixture 2': '#8c564b',     # Brown
             'Mixture 3': '#ff6b35',     # Orange-Red
             'Mixture 4': '#17becf',     # Cyan
             'PM 6 +': '#2ca02c',     # Green (keep existing)
             'PM 6 -': '#1f77b4'      # Blue (keep existing)
            }

pair_comb_data_drop = pair_comb_data.drop(pair_comb_data[pair_comb_data['Name'].isin(names_to_remove_5)].index)
pair_comb_data_drop['Name'] = pair_comb_data_drop['Name'].replace(replace_dict_5)

pair_plot = pr.plot_curves(pair_comb_data_drop, palette=color_map_5);
pair_plot.set(ylim=(0, 1400));

## Steady state
Bar graph of steady-state endpoint of each sample. Steady state is calculated as the maximum fluorescence value over a 3-sample rolling average on the data.

In [ ]:
#| label: fig:day3-steadystate-pairwise

color_map_list = [
    color_map_1,
    color_map_2,
    color_map_3,
    color_map_4,
    color_map_5
]

color_map_list

flat_color_map = {k: v for color_map in color_map_list for k, v in color_map.items()}

pm_keys = ['PM 6 +', 'PM 6 -']
ordered_color_map = {k: v for k, v in flat_color_map.items() if k not in pm_keys}
ordered_color_map.update({k: flat_color_map[k] for k in pm_keys if k in flat_color_map})

data_list_concat = pd.concat([
    argsrs_comb_data_drop,
    t7rnap_comb_data_drop,
    if2_comb_data_drop,
    eftu_comb_data_drop,
    pair_comb_data_drop,
])

pr.plot_steadystate(data_list_concat, order=list(ordered_color_map.keys()), palette=ordered_color_map);
plt.axhline(y=443, color='black', linestyle='--', alpha=0.5);

# Kinetics Analysis
These functions calculate key kinetic parameters of the time series.

In [ ]:
pr.plot_kinetics(data_drop)

We can also calculate the kinetics and display the parameters as a table.

In [ ]:
pr.kinetic_analysis(data_drop)